# Tutorial on quantEM `Vector` class

This tutorial demonstrates how the quantEM `Vector` module works 

Colin Ophus and Arthur McCray
March 5, 2026

In [1]:
import numpy as np
import quantem as em
from quantem.core.datastructures import Vector 


d:\code\quantem\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Creating a Vector

A `Vector` stores ragged per-cell data on a fixed grid. Each cell holds a variable number of rows, one value per named field.

Create an empty `Vector` with `from_shape`, then assign cell data with `[]`:

In [2]:
Nx, Ny = 20, 30  # fixed-grid dimensions (e.g. scan positions)

v = Vector.from_shape(
    shape=(Nx, Ny),
    fields=("kx", "ky", "intensity"),
    units=("A^-1", "A^-1", "counts"),
    name="diffraction_vectors",
)

# Assign each cell a 2D array of shape (n_rows, num_fields)
rng = np.random.default_rng(42)
for rx in range(Nx):
    for ry in range(Ny):
        n = rng.integers(5, 20)
        phi = rng.random(n) * 2 * np.pi
        r = 10 + rng.standard_normal(n) * 2
        v[rx, ry] = np.column_stack((r * np.cos(phi), r * np.sin(phi), rng.random(n)))

print(v)

quantem.Vector, shape=(20, 30), name=diffraction_vectors
  fields = ['kx', 'ky', 'intensity']
  units: ['A^-1', 'A^-1', 'counts']


Alternatively, create from existing nested data with `from_data`. Cells can be lists, tuples, or arrays:

In [3]:
v_small = Vector.from_data(
    data=[
        np.array([[1.0, 2.0], [3.0, 4.0]]),   # cell 0: 2 rows
        np.array([[5.0, 6.0], [7.0, 8.0], [9.0, 10.0]]),  # cell 1: 3 rows
    ],
    fields=["x", "y"],
    units=["m", "m"],
    name="example",
)
print(v_small)
print("\ncell 0:\n", v_small[0].array)
print("\ncell 1:\n", v_small[1].array)

quantem.Vector, shape=(2,), name=example
  fields = ['x', 'y']
  units: ['m', 'm']

cell 0:
 [[1. 2.]
 [3. 4.]]

cell 1:
 [[ 5.  6.]
 [ 7.  8.]
 [ 9. 10.]]


## Properties

In [4]:
print("shape:      ", v.shape)
print("fields:     ", v.fields)
print("units:      ", v.units)
print("num_fields: ", v.num_fields)
print("num_cells:  ", v.num_cells)
print("total_rows: ", v.total_rows)
print("dtype:      ", v.dtype)
print("name:       ", v.name)

shape:       (20, 30)
fields:      ['kx', 'ky', 'intensity']
units:       ['A^-1', 'A^-1', 'counts']
num_fields:  3
num_cells:   600
total_rows:  7243
dtype:       float64
name:        diffraction_vectors


## Fixed-grid indexing

`[]` selects along the fixed-grid axes — the same as NumPy indexing on a regular array. It always returns a `Vector` view over shared storage.

- Integer index → 0D `Vector`; access the cell array with `.array`
- Slice/fancy index → sub-grid `Vector`
- `.flatten()` concatenates all selected cells row-wise into a 2D NumPy array

In [5]:
# 0D selection → use .array to get the NumPy array for that cell
cell = v[0, 0]
print("cell shape:", cell.shape)       # () means scalar / 0D
print("cell array:\n", cell.array)

# Slice along one or both axes → returns a sub-grid Vector
row = v[0, :]   # first row, all columns
print("\nrow shape:", row.shape)

# Flatten all selected cells into one 2D array
flat = v[0, :].flatten()
print("flattened shape:", flat.shape)   # (total_rows, num_fields)

# Copy data from one region to another (write-through)
v[1, :] = v[0, :]

cell shape: ()
cell array:
 [[ -8.68517809   3.50970304   0.82276161]
 [  6.28492484  -7.73490805   0.4434142 ]
 [ -2.69304819  -7.84451847   0.22723872]
 [  9.75950493   6.55906592   0.55458479]
 [ 11.42029689  -1.76304781   0.06381726]
 [  0.70859257 -10.10725307   0.82763117]]

row shape: (30,)
flattened shape: (384, 3)


## Field selection & arithmetic

`select_fields(...)` returns a **write-through view** over a subset of fields. Changes made through the view are reflected in the parent `Vector`.

Arithmetic operators (`+`, `-`, `*`, `/`, `**`, `%`, `//`, unary `-`, `abs`) all work on `Vector` objects and return new `Vector` instances. In-place operators (`+=`, `*=`, etc.) modify the backing data directly.

In [6]:
v2 = v.copy() 
kx = v2.select_fields("kx")
ky = v2.select_fields("ky")
intensity = v2.select_fields("intensity")

# In-place: modifies v directly
kx += 16
ky += 16

# Arithmetic between field views
r_squared = kx**2 + ky**2        # new Vector, field value is "kx" same as kx vector
r_squared.rename_fields({"kx": "r_squared"})
r_squared.name = "r_squared"
print(r_squared)
print("r² first 5 values:", r_squared.flatten()[:5, 0])

# Scale intensity in-place by a per-row factor
scale = kx.flatten() / r_squared.flatten()
intensity.set_flattened(intensity.flatten() * scale)

# Assign using [...]
kx[...] = np.abs(kx.flatten())   # equivalent to abs(kx)[...] = ...

print("\nv fields after modifications:", v2.fields)
print("kx range: [{:.2f}, {:.2f}]".format(kx.flatten().min(), kx.flatten().max()))

quantem.Vector, shape=(20, 30), name=r_squared
  fields = ['r_squared']
  units: ['A^-1']
r² first 5 values: [ 434.1351322   564.92961994  243.58684533 1172.46354937  954.56348915]

v fields after modifications: ['kx', 'ky', 'intensity']
kx range: [0.40, 31.14]


## NumPy ufunc support

NumPy elementwise ufuncs that return a numpy array work directly on `Vector` objects via `__array_ufunc__`. Multi-output ufuncs (e.g. `np.modf`) return tuples of `Vector`. Reduction-based functions (such as `np.mean(arr)` or `np.sum(arr)`) do not work. 

In [7]:
kx = v.select_fields("kx")
ky = v.select_fields("ky")
intensity = v.select_fields("intensity")
print("kx[:2]:\n", kx.flatten()[:2])

print("\nsin(kx)[:2]:\n", np.sin(kx).flatten()[:2])
print("\nmaximum(kx, 10)[:2]:\n", np.maximum(kx, 10).flatten()[:2]) # type: ignore[call-overload]

# Multi-output: modf returns (fractional, integer) as a tuple of Vectors
frac, whole = np.modf(kx)
print("\nmodf frac[:2]:", frac.flatten()[:2, 0])
print("modf whole[:2]:", whole.flatten()[:2, 0])

kx[:2]:
 [[-8.68517809]
 [ 6.28492484]]

sin(kx)[:2]:
 [[-0.67399238]
 [ 0.00173953]]

maximum(kx, 10)[:2]:
 [[10.]
 [10.]]

modf frac[:2]: [-0.68517809  0.28492484]
modf whole[:2]: [-8.  6.]


## Row-wise updates with `set_flattened`

`set_flattened(values)` writes back into all selected cells without changing per-cell row counts. This is the natural companion to `flatten()` for applying row-wise NumPy transforms.

In [8]:
v2 = v.copy()
kx = v2.select_fields("kx")
ky = v2.select_fields("ky")

# Zero-out kx for any row within radius 2 of the origin
mask = (kx.flatten()**2 + ky.flatten()**2) < 100  # shape (total_rows, 1)
kx.set_flattened(np.where(mask, 0.0, kx.flatten()))

print(f"rows zeroed: {mask.sum()} / {v2.total_rows}")

rows zeroed: 3599 / 7229


## Schema operations

`add_fields` and `remove_fields` modify the field schema for the whole `Vector`. `append_rows` adds rows to a single cell.

In [9]:
kx = v.select_fields("kx")
ky = v.select_fields("ky")

# Add a derived field with initial values
v.add_fields("r", values=np.sqrt(kx**2 + ky**2), units="A^-1")
print("fields after add:", v.fields)

# Remove it again
v.remove_fields("r")
print("fields after remove:", v.fields)

# Append rows to a single cell
before = v[0, 0].array.shape[0]
v.append_rows((0, 0), np.array([[1.0, 2.0, 0.5]]))
after = v[0, 0].array.shape[0]
print(f"\ncell [0,0] rows: {before} → {after}")

fields after add: ['kx', 'ky', 'intensity', 'r']
fields after remove: ['kx', 'ky', 'intensity']

cell [0,0] rows: 6 → 7


## File I/O

`Vector` can be saved and loaded using the standard quantEM I/O interface.

In [10]:
import tempfile

with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
    path = tmp.name

try:
    v.save(path, mode="o")
    v_loaded = em.io.load(path)

    # Verify round-trip
    print("loaded:", v_loaded)
    np.testing.assert_array_equal(v[3, 5].array, v_loaded[3, 5].array)
    print("Round-trip OK")
finally:
    import os
    if os.path.exists(path):
        os.remove(path)

loaded: quantem.Vector, shape=(20, 30), name=diffraction_vectors
  fields = ['kx', 'ky', 'intensity']
  units: ['A^-1', 'A^-1', 'counts']
Round-trip OK


-- end notebook --